# Kapitel 18.7 — PostgreSQL in Python

## Lernziele
- PostgreSQL mit Python verwenden.
- Die Bibliothek `psycopg2` nutzen.
- Erweiterte Funktionen nutzen.

## Theorie

PostgreSQL ist eine robuste Open-Source-Datenbank mit vielen erweiterten Funktionen. `psycopg2` ist das Standard-Python-Modul dafür.

## Installation

```bash
pip install psycopg2-binary
```

In [ ]:
import psycopg2

# Verbindung erstellen
conn = psycopg2.connect(
    host="localhost",
    database="schule",
    user="postgres",
    password="passwort",
    port="5432"
)
cursor = conn.cursor()

# Datenbank-Version prüfen
cursor.execute("SELECT version();")
print(cursor.fetchone())

conn.close()

## Beispiel 1
Tabellen mit Constraints erstellen.

In [ ]:
import psycopg2

conn = psycopg2.connect(
    host="localhost",
    database="schule",
    user="postgres",
    password="passwort"
)
cursor = conn.cursor()

# Tabelle mit mehr Constraints
cursor.execute('''
    CREATE TABLE IF NOT EXISTS schüler (
        id SERIAL PRIMARY KEY,
        name VARCHAR(100) NOT NULL,
        klasse VARCHAR(10),
        alter INT CHECK (alter > 0 AND alter < 100),
        erstellt TIMESTAMP DEFAULT CURRENT_TIMESTAMP
    )
''')

conn.commit()
conn.close()

## Beispiel 2
Daten einfügen und abrufen.

In [ ]:
import psycopg2

conn = psycopg2.connect(
    host="localhost",
    database="schule",
    user="postgres",
    password="passwort"
)
cursor = conn.cursor()

# Daten einfügen
cursor.execute(
    "INSERT INTO schüler (name, klasse, alter) VALUES (%s, %s, %s)",
    ('Anna', '10A', 15)
)

# Mehrere Datensätze
daten = [
    ('Beni', '10B', 15),
    ('Carla', '10A', 16),
    ('David', '11A', 17)
]
cursor.executemany(
    "INSERT INTO schüler (name, klasse, alter) VALUES (%s, %s, %s)",
    daten
)

# Abrufen
cursor.execute("SELECT * FROM schüler WHERE klasse = %s", ('10A',))
print(cursor.fetchall())

conn.commit()
conn.close()

## Praxisbeispiel
Mit Fehlerbehandlung arbeiten.

In [ ]:
import psycopg2
from psycopg2 import sql

class PostgreSQLDB:
    def __init__(self, **kwargs):
        try:
            self.conn = psycopg2.connect(**kwargs)
            self.cursor = self.conn.cursor()
        except Exception as e:
            print(f"Fehler: {e}")
    
    def execute(self, query, params=None):
        try:
            self.cursor.execute(query, params)
            self.conn.commit()
        except Exception as e:
            print(f"Fehler: {e}")
            self.conn.rollback()
    
    def fetch_all(self, query, params=None):
        try:
            self.cursor.execute(query, params)
            return self.cursor.fetchall()
        except Exception as e:
            print(f"Fehler: {e}")
            return []
    
    def close(self):
        self.conn.close()

db = PostgreSQLDB(
    host="localhost",
    database="schule",
    user="postgres",
    password="passwort"
)
print(db.fetch_all("SELECT * FROM schüler"))
db.close()

## Häufige Fehler
- Keine Fehlerbehandlung
- Keine Rollback bei Fehlern
- Verbindung nicht geschlossen

## Best Practice
Nutze Context Manager und Exception Handling für robuste Datenbank-Code.

## Zusammenfassung
PostgreSQL mit Python ist ideal für professionelle und komplexe Anwendungen.